# Differentiable inference of ONNX model

This demo shows the full "ONNX -> differentiable C++" path, driven entirely from Python with no external dependencies (no onnx, onnxruntime, torch, tensorflow, ...):

   1. Parse an .onnx model and *generates* plain C++ inference
      code (a header with the weights embedded).
   2. ROOT's interpreter (cling) JIT-compiles that generated code.
   3. Clad, a source-transformation automatic-differentiation plugin,
      differentiates the generated forward pass to give an exact analytic
      gradient of a scalar function of the network output.

What makes this technology special is that we can deploy ONNX models in C++ environments and get their **gradients for free**. The gradients can come in very handy for example if your ONNX model is part of a larger computation for a domain specific loss function that you want to minimize.

Alternatives concidered:

* **LibTorch**
    * Full AD support, but significant deployment cost
* **ONNXRuntime** (with gradients baked into the graph)
    * ONNXRuntime has no AD, but you could export gradient graphs
    * The user would have to manage gradient artifacts explicitly:
error prone non-transparent AD where you lock in your gradients at export time
* **Hand coded C++**
    * Will work well but hard to scale
* **C++ Code generation plus AD with Clad**
    * No extra dependency and transparent AD with Clad
    * Natural fit for our ecosystem

In [ ]:
import os

import ROOT
import numpy as np

from inference_helpers import generate_inference_code, jit_code_and_gradient

Convert the ONNX model to C++:

In [ ]:
# A small checked-in test model: a 2-layer MLP with sigmoid activations
# (Gemm -> Sigmoid -> Gemm -> Sigmoid), input shape [2, 24], output [2, 12].
onnx_file = os.path.join("LinearWithSigmoid.onnx")
model_name = "LinearWithSigmoid"

generated_code, N_IN = generate_inference_code(onnx_file)

In [ ]:
# Print for visual inspection
print(generated_code)

JIT C++ code, generate gradient, and also JIT the gradient:

In [ ]:
value_func, gradient_func, N_OUT = jit_code_and_gradient(model_name, generated_code, N_IN)

Evaluate forward pass and gradient:

In [ ]:
# --- Use it from Python with numpy arrays -----------------------------------
rng = np.random.default_rng(0)
x = rng.standard_normal(N_IN).astype(np.float32)

val = value_func(x)
print(val)

grad = np.zeros(N_IN, dtype=np.float32)
gradient_func(x, grad)
print(grad)

Calculate finite differences for comparison:

In [ ]:
# --- Verify the analytic gradient against central finite differences --------
eps = 1e-3
fd = np.empty(N_IN, dtype=np.float32)
for i in range(N_IN):
    xp, xm = x.copy(), x.copy()
    xp[i] += eps
    xm[i] -= eps
    fd[i] = (value_func(xp) - value_func(xm)) / (2 * eps)

max_err = float(np.max(np.abs(grad - fd)))

Summary:

In [ ]:
print(f"Differentiable SOFIE inference for '{model_name}' ({N_IN} inputs -> {N_OUT} outputs)")
print(f"  f(x) = sum(outputs)            = {val:.6f}")
print(f"  Clad gradient        [:4]      = {np.array2string(grad[:4], precision=5)}")
print(f"  Finite-diff gradient [:4]      = {np.array2string(fd[:4],   precision=5)}")
print(f"  max |Clad - finite difference| = {max_err:.2e}")
assert max_err < 2e-3, "Clad gradient does not match finite differences!"
print("  OK: Clad analytic gradient matches finite differences.")

If you need it, you can look at the generated C++ inference and gradient code via the interpreter.

In [ ]:
%%cpp

TMVA_SOFIE_LinearWithSigmoid::doInfer

In [ ]:
%%cpp

TMVA_SOFIE_LinearWithSigmoid::doInfer_pullback